Titanic: Machine Learning from Disaster

In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


WITH JAX 

In [2]:
import jax
import jax.numpy as jnp
from jax import grad, jit
import pandas as pd
import time

# -----------------------------
# Load Data
# -----------------------------
path = "/kaggle/input/competitions/titanic/"
train = pd.read_csv(path + "train.csv")

features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare"]

train["Age"] = train["Age"].fillna(train["Age"].median())
train["Fare"] = train["Fare"].fillna(train["Fare"].median())
train["Sex"] = train["Sex"].map({"male": 0, "female": 1})

X = train[features].values
y = train["Survived"].values.reshape(-1, 1)

# Normalize
X_mean = X.mean(axis=0)
X_std = X.std(axis=0) + 1e-8
X = (X - X_mean) / X_std

X = jnp.array(X)
y = jnp.array(y)

# -----------------------------
# Model
# -----------------------------
def sigmoid(z):
    return 1 / (1 + jnp.exp(-z))

def predict(w, X):
    return sigmoid(jnp.dot(X, w))

def loss_fn(w, X, y):
    p = predict(w, X)
    return -jnp.mean(y * jnp.log(p + 1e-8) + (1 - y) * jnp.log(1 - p + 1e-8))

grad_fn = grad(loss_fn)

@jit
def update(w, X, y, lr):
    return w - grad_fn(w, X, y) * lr

@jit
def train_loop(w, X, y, lr, steps):
    def body(i, w):
        return update(w, X, y, lr)
    return jax.lax.fori_loop(0, steps, body, w)

# -----------------------------
# Training + Timing
# -----------------------------
w = jnp.zeros((X.shape[1], 1))

start = time.time()
w = train_loop(w, X, y, lr=0.05, steps=5000)
end = time.time()

# -----------------------------
# Accuracy
# -----------------------------
preds = predict(w, X)
acc = jnp.mean((preds > 0.5) == y)

print("JAX Accuracy:", acc)
print("JAX Training Time:", end - start, "seconds")

JAX Accuracy: 0.7867565
JAX Training Time: 0.22301220893859863 seconds


WITH OUT JAX

In [3]:
import pandas as pd
import numpy as np
import time

# -----------------------------
# Load data
# -----------------------------
path = "/kaggle/input/competitions/titanic/"
train = pd.read_csv(path + "train.csv")

features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare"]

train["Age"] = train["Age"].fillna(train["Age"].median())
train["Fare"] = train["Fare"].fillna(train["Fare"].median())
train["Sex"] = train["Sex"].map({"male": 0, "female": 1})

X = train[features].values
y = train["Survived"].values.reshape(-1, 1)

# -----------------------------
# Normalize
# -----------------------------
X_mean = X.mean(axis=0)
X_std = X.std(axis=0) + 1e-8
X = (X - X_mean) / X_std

# -----------------------------
# Sigmoid
# -----------------------------
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# -----------------------------
# Initialize weights
# -----------------------------
w = np.zeros((X.shape[1], 1))

lr = 0.05
steps = 5000

# -----------------------------
# Training (manual gradient descent)
# -----------------------------
start = time.time()

for i in range(steps):
    z = np.dot(X, w)
    preds = sigmoid(z)

    # gradient of loss
    error = preds - y
    grad = np.dot(X.T, error) / len(X)

    w -= lr * grad

end = time.time()

# -----------------------------
# Prediction + Accuracy
# -----------------------------
final_preds = sigmoid(np.dot(X, w))
final_labels = (final_preds > 0.5).astype(int)

acc = np.mean(final_labels == y)

print("NumPy Accuracy:", acc)
print("NumPy Training Time:", end - start)

NumPy Accuracy: 0.7867564534231201
NumPy Training Time: 0.15100598335266113
